# 07 · Views and Materialized Views

> **Module 5 · Views & Materialized Views · Lectures 5.3–5.4**

In this notebook you will:

- Create logical views and understand when they expand
- Build authorized views for row-level security
- Create materialized views and compare their cost to a view
- Use the MV + view pattern to get around the single-base-table rule
- Refresh MVs incrementally

---

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. A logical view

A view stores a **query**, not data. It is expanded at query time, so it costs the
same to query as the underlying table — but it gives you a stable, reusable name
and a place to enforce security.

In [ ]:
view = bigquery.Table(f"{PROJECT_ID}.curated.v_active_orders")
view.view_query = f"""
    SELECT order_id, customer_id, order_date, total_amount, region
    FROM `{PROJECT_ID}.curated.orders_clustered`
    WHERE status = 'active'
"""
view.description = "Active orders only — the stable interface for analysts"
view.labels = {"tier": "silver", "owner": "analytics-eng"}
view = client.create_view(view, exists_ok=True)
print(f"✓ {view.full_table_id}")

df = run_to_dataframe(f"SELECT * FROM `{view.full_table_id}` LIMIT 10")
display(df)

In [ ]:
# A view costs the same as the base table — it is expanded at query time
VIEW_QUERY = f"SELECT COUNT(*) FROM `{PROJECT_ID}.curated.v_active_orders`"
BASE_QUERY = f"SELECT COUNT(*) FROM `{PROJECT_ID}.curated.orders_clustered` WHERE status = 'active'"

view_cost = estimate_cost(VIEW_QUERY)
base_cost = estimate_cost(BASE_QUERY)

print(f"via view: {view_cost['bytes_processed']:,} bytes → ${view_cost['estimated_cost_usd']:.6f}")
print(f"via base: {base_cost['bytes_processed']:,} bytes → ${base_cost['estimated_cost_usd']:.6f}")

print("\nIdentical — the view adds no storage cost and no query discount.")

## 2. Views over joins

This is the classic use of a view: hide a join behind a stable interface.

In [ ]:
# A customers reference table
customers_schema = [
    bigquery.SchemaField("customer_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("customer_name", "STRING"),
    bigquery.SchemaField("segment", "STRING"),
    bigquery.SchemaField("country", "STRING"),
]

customers = bigquery.Table(f"{PROJECT_ID}.curated.customers", schema=customers_schema)
customers = client.create_table(customers, exists_ok=True)

customer_rows = [
    {"customer_id": f"c{i:04d}", "customer_name": f"Customer {i}",
     "segment": ["enterprise", "midmarket", "smb"][i % 3], "country": "GB"}
    for i in range(1, 51)
]
errors = client.insert_rows_json(client.dataset("curated").table("customers"), customer_rows)
print("Errors:", errors if errors else "none ✅")

In [ ]:
joined_view = bigquery.Table(f"{PROJECT_ID}.curated.v_orders_with_customer")
joined_view.view_query = f"""
    SELECT
      o.order_id,
      o.order_date,
      o.total_amount,
      o.status,
      o.region,
      c.customer_name,
      c.segment,
      c.country
    FROM `{PROJECT_ID}.curated.orders_clustered` AS o
    LEFT JOIN `{PROJECT_ID}.curated.customers` AS c
      ON o.customer_id = c.customer_id
"""
joined_view = client.create_view(joined_view, exists_ok=True)
print(f"✓ {joined_view.full_table_id}")

df = run_to_dataframe(f"SELECT * FROM `{joined_view.full_table_id}` LIMIT 10")
display(df)

## 3. Authorized views — row-level security

An authorized view runs with the **view owner's** permissions and adds a row
predicate. Grant the consumer access to the **view only** — not the base table.

In [ ]:
auth_view = bigquery.Table(f"{PROJECT_ID}.curated.v_emea_orders")
auth_view.view_query = f"""
    SELECT *
    FROM `{PROJECT_ID}.curated.orders_clustered`
    WHERE region = 'EMEA'
"""
auth_view.description = "EMEA orders only — grant analysts this, not the base table"
auth_view = client.create_view(auth_view, exists_ok=True)

print(f"✓ {auth_view.full_table_id}")

# Grant a user access to the VIEW ONLY (uncomment with a real principal)
# client.query(
#     f"""GRANT `roles/bigquery.dataViewer`
#        ON VIEW `{PROJECT_ID}.curated.v_emea_orders`
#        TO \'user:analyst@yourcompany.com\'"""
# )

df = run_to_dataframe(
    f"SELECT region, COUNT(*) AS n FROM `{auth_view.full_table_id}` GROUP BY region"
)
display(df)

In [ ]:
# A view that filters by the caller's identity
session_view = bigquery.Table(f"{PROJECT_ID}.curated.v_my_rows")
session_view.view_query = f"""
    SELECT *
    FROM `{PROJECT_ID}.curated.orders_clustered`
    WHERE customer_id = SESSION_USER()
"""
session_view = client.create_view(session_view, exists_ok=True)
print(f"✓ {session_view.full_table_id} — filters on SESSION_USER()")

## 4. Materialized views

An MV stores the **result** of a query. It is pre-computed, so querying it is
dramatically cheaper — at the cost of storage and refresh.

> ⚠️ **An MV is based on exactly ONE table.** That single rule drives the whole
> design of any serving layer.

In [ ]:
mv = bigquery.Table(f"{PROJECT_ID}.mart.mv_daily_revenue")
mv.mview_query = f"""
    SELECT
      order_date,
      region,
      channel,
      COUNT(*) AS order_count,
      COUNT(DISTINCT customer_id) AS customer_count,
      SUM(total_amount) AS revenue,
      SAFE_DIVIDE(SUM(total_amount), COUNT(*)) AS aov
    FROM `{PROJECT_ID}.curated.orders_clustered`
    GROUP BY order_date, region, channel
"""
mv.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY, field="order_date"
)
mv.clustering_fields = ["region"]
mv.description = "Daily revenue by region and channel"
mv.labels = {"tier": "gold", "owner": "analytics-eng"}

mv = client.create_table(mv, exists_ok=True)
print(f"✓ {mv.full_table_id}")

df = run_to_dataframe(f"SELECT * FROM `{mv.full_table_id}` ORDER BY order_date DESC LIMIT 10")
display(df)

In [ ]:
# The payoff: compare querying the MV against querying the base table
MV_QUERY = f"""
SELECT order_date, region, channel, revenue
FROM `{PROJECT_ID}.mart.mv_daily_revenue`
WHERE order_date = DATE '2026-01-15'
"""

BASE_QUERY_EQUIVALENT = f"""
SELECT order_date, region, channel, SUM(total_amount) AS revenue
FROM `{PROJECT_ID}.curated.orders_clustered`
WHERE order_date = DATE '2026-01-15'
GROUP BY order_date, region, channel
"""

mv_est = estimate_cost(MV_QUERY)
base_est = estimate_cost(BASE_QUERY_EQUIVALENT)

print(f"via MV:  {mv_est['bytes_processed']:>12,} bytes  → ${mv_est['estimated_cost_usd']:.6f}")
print(f"via base: {base_est['bytes_processed']:>12,} bytes  → ${base_est['estimated_cost_usd']:.6f}")

if base_est["bytes_processed"]:
    saving = 1 - (mv_est["bytes_processed"] / base_est["bytes_processed"])
    print(f"\nSaving: {saving:.1%} fewer bytes scanned")

## 5. MV metadata and refresh

An MV refreshes automatically in the background. `max_staleness` controls how far
behind it may lag.

In [ ]:
# Inspect the MV
df = run_to_dataframe(
    f"""
    SELECT
      table_name,
      refresh_interval_minutes,
      max_staleness,
      last_refresh_time,
      creation_time
    FROM `{PROJECT_ID}.`region-us`.INFORMATION_SCHEMA.MATERIALIZED_VIEWS
    WHERE table_name = 'mv_daily_revenue'
    """
)
display(df)

In [ ]:
# Manual full refresh
run_query(
    f"CALL bq.refresh_materialized_view('{PROJECT_ID}.mart.mv_daily_revenue')"
)
print("✓ Refreshed the MV")

# Incremental refresh — only the last 3 days. Much cheaper on a large table.
run_query(
    f"""
    CALL bq.refresh_materialized_view(
      '{PROJECT_ID}.mart.mv_daily_revenue',
      INTERVAL 3 DAY
    )
    """
)
print("✓ Incrementally refreshed the last 3 days")

In [ ]:
# Set the refresh policy with the Python client
table = client.get_table(f"{PROJECT_ID}.mart.mv_daily_revenue")
table.labels = {**(table.labels or {}), "refresh": "hourly"}
table = client.update_table(table, update_mode="update_fields")
print(f"✓ Labels: {table.labels}")

print("\nFor max_staleness and enable_refresh, use SQL:")
print("  ALTER MATERIALIZED VIEW `p.ds.mv` SET OPTIONS (max_staleness = INTERVAL 1 HOUR)")

## 6. The MV + view pattern

The MV + view pattern is the standard workaround for the single-base-table rule.
**Step 1:** a view that does the join. **Step 2:** an MV over the view.

In [ ]:
# Step 1: a view that performs the join
join_view = bigquery.Table(f"{PROJECT_ID}.mart.v_orders_with_region")
join_view.view_query = f"""
    SELECT
      o.order_id,
      o.order_date,
      o.total_amount,
      o.channel,
      c.segment,
      c.country
    FROM `{PROJECT_ID}.curated.orders_clustered` AS o
    JOIN `{PROJECT_ID}.curated.customers` AS c
      ON o.customer_id = c.customer_id
"""
join_view = client.create_view(join_view, exists_ok=True)
print(f"✓ Step 1: {join_view.full_table_id}")

# Step 2: an MV over the view — the view counts as ONE base table
mv2 = bigquery.Table(f"{PROJECT_ID}.mart.mv_revenue_by_segment")
mv2.mview_query = f"""
    SELECT
      order_date,
      segment,
      SUM(total_amount) AS revenue,
      COUNT(*) AS order_count
    FROM `{PROJECT_ID}.mart.v_orders_with_region`
    GROUP BY order_date, segment
"""
mv2.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY, field="order_date"
)
mv2 = client.create_table(mv2, exists_ok=True)
print(f"✓ Step 2: {mv2.full_table_id}")

df = run_to_dataframe(f"SELECT * FROM `{mv2.full_table_id}` ORDER BY order_date DESC LIMIT 10")
display(df)

## 7. Demonstrate the single-base-table restriction

This query fails. It is the constraint that shapes every serving-layer design.

In [ ]:
try:
    run_query(
        f"""
        CREATE MATERIALIZED VIEW `{PROJECT_ID}.mart.mv_will_fail` AS
        SELECT o.order_id, c.segment
        FROM `{PROJECT_ID}.curated.orders_clustered` AS o
        JOIN `{PROJECT_ID}.curated.customers` AS c ON o.customer_id = c.customer_id
        GROUP BY 1, 2
        """
    )
except Exception as exc:
    message = str(exc)
    print("✅ Expected failure:")
    # Print the informative part of the error
    for line in message.split("\n"):
        if any(k in line for k in ("single", "base table", "Materialized", "Error", "error")):
            print(f"   {line.strip()}")
    print("\n   Use the MV + view pattern instead (see the previous cell).")

## 8. A complete serving layer

Three layers, each with a clear job:

In [ ]:
# Layer 1: the base (curated) table — the source of truth
# Layer 2: a view — joins and row-level security, no storage cost
# Layer 3: an MV — pre-aggregated, cheap to query, dashboard-ready

LAYERS = {
    "base (curated)":   f"`{PROJECT_ID}.curated.orders_clustered`",
    "view (join)":      f"`{PROJECT_ID}.curated.v_orders_with_customer`",
    "mv (aggregate)":   f"`{PROJECT_ID}.mart.mv_daily_revenue`",
}

print(f"{'layer':<20} {'storage cost':<15} {'query cost':<14} {'freshness'}")
print("-" * 62)
print(f"{'base (curated)':<20} {'full':<15} {'full':<14} {'always'}")
print(f"{'view (join)':<20} {'free':<15} {'full':<14} {'always'}")
print(f"{'mv (aggregate)':<20} {'storage':<15} {'tiny':<14} {'up to max_staleness'}")

In [ ]:
# Storage cost of the MV
df = run_to_dataframe(
    f"""
    SELECT
      table_name,
      table_type,
      ROUND(size_bytes / 1024, 1) AS kb,
      row_count
    FROM `{PROJECT_ID}.`region-us`.INFORMATION_SCHEMA.TABLE_STORAGE
    WHERE table_schema = 'mart'
    """
)
display(df)

## 9. The rules

**Materialized view restrictions:**

- Exactly **one** base table (a view counts as one)
- No subqueries
- Cannot `GROUP BY` an aggregate
- No direct `JOIN` — use the MV + view pattern
- Cannot reference a parameterized view
- The base table's partition filter is inherited

**View rules:**

- No storage cost, no refresh cost
- Always fresh — expands at query time
- Can be time-travelled
- Can be authorized for row-level security
- Cannot be the base of another view in some contexts — prefer CTAS for chained logic

## 🔑 Takeaways

- A **view** stores a query. Zero storage cost, always fresh, same query cost as the
  base table.
- An **authorized view** is how you implement row-level security — grant the view,
  not the table.
- A **materialized view** stores the result. Much cheaper to query, costs storage,
  can be stale.
- **An MV has exactly one base table** — the MV + view pattern is the workaround.
- `max_staleness` controls freshness; `refresh_materialized_view` gives manual control.
- Build a serving layer: **base → view (joins, security) → MV (dashboards)**.

---

## 📚 Next

[08 · Cost Analysis](./08-cost-analysis.ipynb)